# 4. Classifying Images

**Rubric:**
- `classifier(images_dir + users_key, model)` -- `images_dir` is prepended to each key before calling `classifier()`
- Output is lowercased and stripped of whitespace
- Results are verified and stored correctly; 1 is appended for a correct label, 0 for an incorrect one

> This build has no internet access to download the real pretrained ResNet/AlexNet/VGG weights, so `classifier.py` returns pre-recorded labels from `demo_data.py` instead of running a real CNN forward pass. Every other line of logic here -- the path construction, the lowercasing, the match check, the 1/0 bookkeeping -- is exactly what runs against a real classifier; only the inside of `classifier()` itself is swapped out. See the note at the top of `classifier.py`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # so we can import the project's .py files


In [2]:
import inspect
from classify_images import classify_images
print(inspect.getsource(classify_images))

def classify_images(images_dir, results_dic, model):
    """
    Runs classifier() on every image in results_dic and appends the
    classifier's label and a match flag to that image's entry.

    Parameters:
     images_dir - path to the folder of pet images, ending in '/'
     results_dic - dictionary from get_pet_labels(); this function extends
                   each value list in place with
                   [pet_label, classifier_label, is_match]
     model - CNN model architecture to use: 'resnet', 'alexnet' or 'vgg'

    Returns:
     None - results_dic is mutated in place
    """
    for users_key in results_dic:
        # images_dir already ends in '/' (see get_input_args' default), so
        # the path is built by simple concatenation.
        classifier_label = classifier(images_dir + users_key, model)
        classifier_label = classifier_label.lower().strip()

        pet_label = results_dic[users_key][0]

        # A match only counts if pet_label appears in classifier

## Running it end to end

In [3]:
from get_pet_labels import get_pet_labels

images_dir = '../pet_images/'
results_dic = get_pet_labels(images_dir)
classify_images(images_dir, results_dic, 'vgg')

# Each value is now [pet_label, classifier_label, is_match]
import itertools
for filename, value in itertools.islice(results_dic.items(), 8):
    print(f'{filename:24s} {value}')

Poodle_02.jpg            ['poodle', 'poodle', 1]
Boxer_02.jpg             ['boxer', 'boxer', 1]
Chihuahua_03.jpg         ['chihuahua', 'toy terrier', 0]
Boxer_01.jpg             ['boxer', 'boxer', 1]
Collie_02.jpg            ['collie', 'collie', 1]
Saint_bernard_01.jpg     ['saint bernard', 'saint bernard', 1]
Great_dane_02.jpg        ['great dane', 'great dane', 1]
Poodle_03.jpg            ['poodle', 'poodle', 1]


## Checking each rubric point directly

**`images_dir + users_key`** -- confirm the exact path handed to `classifier()`:

In [4]:
from classifier import classifier

sample_key = 'Beagle_01.jpg'
path_that_gets_built = images_dir + sample_key
print('Path passed to classifier():', path_that_gets_built)
print('classifier() returns:', repr(classifier(path_that_gets_built, 'vgg')))

Path passed to classifier(): ../pet_images/Beagle_01.jpg
classifier() returns: 'beagle'


**Lowercased and stripped** -- classify_images.py does this immediately after calling classifier():

In [5]:
raw = '  Great Dane  '  # what a real pretrained model's raw label can look like
cleaned = raw.lower().strip()
print(repr(raw), '->', repr(cleaned))

'  Great Dane  ' -> 'great dane'


**1 for a match, 0 for a miss** -- tally them up across the whole result set:

In [6]:
n_match = sum(v[2] for v in results_dic.values())
n_miss = len(results_dic) - n_match
print(f'{n_match} images matched (flagged 1), {n_miss} did not (flagged 0)')

print('\nA mismatch, to see the 0 flag in context:')
for filename, value in results_dic.items():
    if value[2] == 0:
        print(f'  {filename:24s} pet={value[0]!r:20s} classifier={value[1]!r:20s} is_match={value[2]}')

39 images matched (flagged 1), 1 did not (flagged 0)

A mismatch, to see the 0 flag in context:
  Chihuahua_03.jpg         pet='chihuahua'          classifier='toy terrier'        is_match=0
